# 🧠 LeetCode 424: Longest Repeating Character Replacement

**Pattern:** Sliding Window & Dynamic Frequencies

- **Created:** 2026-02-28
- **Focus:** Window logic based on `size - max_freq <= k`
- **Tags:** `string` | `sliding-window` | `hash-table` | `medium` | `citi-prep`

## 📖 Problem Statement

You are given a string `s` and an integer `k`. You can choose any character of the string and change it to any other uppercase English character. You can perform this operation at most `k` times.

Return the length of the longest substring containing the same letter you can get after performing the above operations.

### Example 1:
- **Input:** `s = "ABAB"`, `k = 2`
- **Output:** `4`
- **Explanation:** Replace the two 'A's with two 'B's or vice versa.

### Example 2:
- **Input:** `s = "AABABBA"`, `k = 1`
- **Output:** `4`
- **Explanation:** Replace the one 'A' in the middle with 'B' and form "AABBBBA". The substring "BBBB" has the longest repeating letters, which is 4.

## 🧠 Mental Model & Intuition

Think of a Sliding Window as a team working on a project. 

The team has one clear "Leader" (the character that appears the *most* inside the window). Everyone else on the team who is NOT that leader needs to be "replaced" or fired (using up your budget of `k`).

**The Formula for a Valid Window:**
`Total People in Window (Window Size) - People matching the Leader (Max Frequency) <= Budget (k)`

As long as this is true, the window expands right. The moment you run out of budget (too many non-leaders), you shrink the window from the left.

## 🐢 Brute Force Approach

Check every possible substring, manually count the frequencies within, find the maximum frequency, and check if `length - max_freq <= k`.

```python
from collections import Counter
def characterReplacementBrute(s: str, k: int) -> int:
    max_len = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            sub = s[i:j]
            max_freq = max(Counter(sub).values())
            if len(sub) - max_freq <= k:
                max_len = max(max_len, len(sub))
    return max_len
```
**Time:** $O(N^3)$ | **Space:** $O(N)$

In [ ]:
# Optimal Sliding Window Approach
def characterReplacement(s: str, k: int) -> int:
    count = {} # Tally sheet for characters in current window
    max_freq = 0 # The frequency of the "Leader" in the current window
    l = 0
    max_len = 0
    
    for r in range(len(s)):
        # Add the new right-character to our tally
        count[s[r]] = count.get(s[r], 0) + 1
        
        # Did this new character become the leader?
        max_freq = max(max_freq, count[s[r]])
        
        # Validation Check: 
        # Is the number of non-leaders greater than my replacement budget k?
        window_size = r - l + 1
        if window_size - max_freq > k:
            # We went over budget. Shrink the window from the left.
            count[s[l]] -= 1
            l += 1
            
        # Notice we only update max_len when the window IS valid
        # (Because we use an 'if' instead of 'while' for the shrink, 
        # the window never actually shrinks its total physical width, 
        # it just shifts rightward. `r - l + 1` is strictly non-decreasing).
        max_len = max(max_len, r - l + 1)
        
    return max_len

print("Longest for AABABBA w/ k=1:", characterReplacement("AABABBA", 1))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(N)$. `l` and `r` both only move rightward. Even though we calculate `max_freq`, we do it in $O(1)$ by just comparing the old `max_freq` to the incremented count of the incoming character at `r`.
- **Space Complexity:** $O(26) = O(1)$. The `count` hash map only ever stores uppercase English letters, so its maximum size is a constant 26.

## 🎤 Interview Q&A

### Q1: When `l` shifts right, why don't we decrement `max_freq` if the character we dropped WAS the leader?
**Answer:** This is the most brilliant trick of this specific algorithm. We *don't care* if `max_freq` is technically inaccurate for the *current* window after shrinking. Why? Because we only care about expanding the *longest historical window we've seen so far*. To expand the window size further, we must find a *new* historical maximum frequency that exceeds the old record anyway. Leaving `max_freq` historically inflated doesn't break the forward-looking math. It just prevents smaller sub-optimal windows from acting like they solved the problem.

---

### Q2: Why use an `if` statement for shrinking instead of a `while` loop?
**Answer:** In many sliding window problems, you use `while is_invalid: shrink()`. Here, because we are strictly looking for the *max length*, we can just use an `if` to shift the window evenly to the right (incrementing both `l` and `r` evenly). The window size locks at the current maximum and slides along until it finds a sequence that allows it to grow again. Both achieve $O(N)$ time, but the `if` is a slight micro-optimization.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Max Frequency (`max_freq`)** | The count of the most dominant element in the window. | The "Leader" pattern. |
| **Non-decreasing Window** | A sliding window that only expands or shifts laterally, never shrinking. | Using `if` instead of `while` to shift `l`. |

## 💼 The Citi Narrative

**Context:** Error Tolerance in Data Feeds.

**Scenario:** A market data connection would occasionally drop packets (recorded as an 'X' error string, while healthy ticks were 'A', 'B', etc.). The system could tolerate up to `k=3` dropped packets in a row before triggering a circuit breaker. We needed to calculate the longest historical contiguous run of a specific healthy tick symbol, assuming the error drops were magically "replaced" by healthy ticks.

**Impact:** Doing this analysis historically over billions of ticks was heavily I/O bound. Implementing this precise `length - max_freq <= k` sliding window in PySpark as a custom UDAF (User Defined Aggregate Function) allowed us to stream-process massive batches of market ticks continuously in $O(N)$ time, proving our circuit breaker thresholds were properly calibrated.

In [ ]:
# EXERCISE: Trace the logic with the non-decreasing window.
# When window is ABCD, and k=1. L=A (0), R=D(3).
# size(4) - max_f(1) = 3. 3 > 1 (OVER BUDGET).
# L goes to 1. Window shifts right, size stays 3.
print("Sliding windows that use `if` instead of `while` are brilliant because they just 'carry' their maximum width forward.")

## 🎯 Summary: Key Takeaways

### The Pattern
**Sliding Window & Dynamic Frequencies** — Validating a window using `size - max_freq <= k`.

### When to Use It
- ✅ Error tolerance spans in noisy telemetry feeds.
- ✅ Contiguous streak analysis with allowed error fuzzing.
- ❌ **Don't use when:** Order of replacements matters for multiple characters.

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N^3)$ | $O(N)$ |
| Optimal | $O(N)$ | $O(1)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Sliding Window & Dynamic Frequencies** and you've mastered one of the most common patterns in data engineering interviews. 🚀